In [30]:
import numpy as np
import os
import  matplotlib.image as mpimg
from scipy import misc
from PIL import Image
import matplotlib.pyplot as plt
plt.show()

In [31]:
#load data
path = "orl_faces/s2/2.pgm"
try:  
    img  = Image.open(path) 
except IOError: 
    print("fail")

In [32]:
#hàm convert image sang vector
def img_to_vector(filename):
    img  = Image.open(filename).resize((112, 92)) 
    img_vector = np.array(img).flatten()
    sz = img_vector.shape[0]
    return img_vector

In [33]:
#hàm tiền xử lý: đọc ảnh từ tập dữ liệu
#đưa dữ liệu đọc được vào 2 mảng là train_set và test_set 
#train_set_number và test_set_number là id của những người trong tập train và test
def preprocess(dataset_dir = "./orl_faces"):
    train_set = []
    train_set_number = []
    test_set = []
    test_set_number = []
    for i in range(15):
        person_id = i+1
        for j in range(10):
            if j < 6 :
                path = dataset_dir + '/s'+str(person_id)+'/'+str(j+1)+'.pgm'
                img = img_to_vector(path).astype(np.int64) 
                train_set.append(img)
                train_set_number.append(person_id)
            else : 
                path = dataset_dir + '/s'+str(person_id)+'/'+str(j+1)+'.pgm'
                img = img_to_vector(path).astype(np.int64)
                test_set.append(img)
                test_set_number.append(person_id)
        
    train_set = np.array(train_set)
    train_set_number = np.array(train_set_number)
    test_set = np.array(test_set)
    test_set_number = np.array(test_set_number)
    return train_set.T, train_set_number, test_set.T, test_set_number

In [34]:
train_set,train_set_number,test_set,test_set_number = preprocess()

In [35]:
print(train_set)
print(train_set.shape)

[[ 48  59  39 ...  39 117  39]
 [ 49  62  42 ...  40 120  41]
 [ 47  61  51 ...  39 121  39]
 ...
 [ 47  34  27 ...  32  77  39]
 [ 46  34  27 ...  34  76  38]
 [ 46  33  29 ...  39  72  40]]
(10304, 90)


In [36]:
print(train_set_number)
print(train_set_number.shape)

[ 1  1  1  1  1  1  2  2  2  2  2  2  3  3  3  3  3  3  4  4  4  4  4  4
  5  5  5  5  5  5  6  6  6  6  6  6  7  7  7  7  7  7  8  8  8  8  8  8
  9  9  9  9  9  9 10 10 10 10 10 10 11 11 11 11 11 11 12 12 12 12 12 12
 13 13 13 13 13 13 14 14 14 14 14 14 15 15 15 15 15 15]
(90,)


In [37]:
print(test_set)
print(test_set.shape)

[[ 41  44  41 ...  45 120 120]
 [ 45  43  41 ...  41 120 120]
 [ 46  37  43 ...  39 120 122]
 ...
 [ 36  44  42 ...  71  74  73]
 [ 37  41  43 ...  23  75  75]
 [ 37  36  41 ...  37  80  76]]
(10304, 60)


In [38]:
print(test_set_number)
print(test_set_number.shape)

[ 1  1  1  1  2  2  2  2  3  3  3  3  4  4  4  4  5  5  5  5  6  6  6  6
  7  7  7  7  8  8  8  8  9  9  9  9 10 10 10 10 11 11 11 11 12 12 12 12
 13 13 13 13 14 14 14 14 15 15 15 15]
(60,)


In [39]:
#hàm tìm trung bình
def find_mean(arr):
    M = np.mean(arr, axis = 1)
    return M

In [40]:
#tìm trung bình của ma trận tập train_set
M = find_mean(train_set)
print(M)

[92.02222222 92.2        92.3        ... 59.11111111 58.32222222
 58.37777778]


In [41]:
#tại mỗi giá trị của ma trận tập train_set, ta lấy giá trị đó trừ cho
#giá trị trung bình vừa tìm được ở trên
def mean_normalization(arr):
    M = find_mean(arr)
    x_range = arr.shape[0]
    y_range = arr.shape[1]
    for i in range(x_range):
        for j in range(y_range):
            arr[i][j] -= M[i]
    return arr

In [42]:
#A là ma trận sau khi thực hiện hàm mean_normalize
A = mean_normalization(train_set)
A = A
#print(A)

In [43]:
print(A.shape)

(10304, 90)


In [44]:
#nhân A với ma trận chỉnh vị của nó
L = np.dot(A.T, A)
print(L.shape)
print(L)

(90, 90)
[[11737307  4189593  6693137 ...   -16928 -3068034   253940]
 [ 4189593 23277834  7749643 ...   343607  -807505  1254948]
 [ 6693137  7749643 15920912 ...  -985212 -2769610   986993]
 ...
 [  -16928   343607  -985212 ... 10800058   565599  6508585]
 [-3068034  -807505 -2769610 ...   565599  8300760  -174130]
 [  253940  1254948   986993 ...  6508585  -174130 10387651]]


In [45]:
eigenvalues, eigenvectors = np.linalg.eig(L)
# sort eigenvectors theo giá trị của eigenvalues
print("eigenvalues shape : ", eigenvalues.shape)
print("eigenvalues : ", eigenvalues)
print("eigenvectors shape : ", eigenvectors.shape)
print("eigenvectors : ", eigenvectors)

eigenvalues shape :  (90,)
eigenvalues :  [2.46622225e+08 1.87592314e+08 1.06490032e+08 9.96167729e+07
 7.74039190e+07 6.18342103e+07 4.81998627e+07 3.93425773e+07
 3.46729481e+07 2.88078662e+07 2.48042590e+07 2.08518750e+07
 2.00243550e+07 1.80153283e+07 1.69055804e+07 1.60308129e+07
 1.40933343e+07 1.39149463e+07 1.23432410e+07 1.26268985e+07
 1.06943619e+07 1.00436195e+07 9.48688337e+06 8.25663536e+06
 7.74673045e+06 7.22270069e+06 6.94657494e+06 6.90780245e+06
 6.53541322e+06 6.49562845e+06 6.12183739e+06 5.86882229e+06
 5.64469474e+06 5.12882902e+06 7.48279301e+04 4.96149546e+06
 4.90134763e+06 4.70251305e+06 4.55055322e+06 4.42688955e+06
 4.34223955e+06 7.01087051e+05 4.14440532e+06 4.02017389e+06
 3.86681746e+06 9.93870991e+05 3.69718722e+06 3.71904327e+06
 3.57123132e+06 3.47463926e+06 3.44945948e+06 1.05146822e+06
 1.13294895e+06 1.15836518e+06 1.19789743e+06 3.39227153e+06
 3.28652199e+06 1.28848258e+06 1.24057694e+06 1.33557220e+06
 1.38661548e+06 1.39333011e+06 1.47898524e+

In [46]:
U = []
sz = L.shape[0] #L là kết quả thu được khi nhân A với A.T
#nhân ma trận A với eigenvectors có bậc nằm trong khoảng sz
#kết quả từng phép nhân đó được thêm vào ma trận U
for i in range(sz):
    U.append(np.dot(A,eigenvectors[:,i]))

U = np.array(U)
U = U.T
print(U.shape)
print(U)

(10304, 90)
[[173.333599   159.73761192  26.9240268  ...  -6.11895542  -4.00804302
    7.18853954]
 [168.80571575 164.95341288  23.9299432  ...  -2.29691982  -4.48482313
    5.57234662]
 [169.14951086 161.97074969  27.01075079 ...   1.24622679  -5.09366751
    5.23786331]
 ...
 [-42.59818031  38.18536547  76.06517485 ... -20.05293033   2.16953824
  -10.11410455]
 [-28.09068338  44.61681005  63.75939431 ... -17.54344629  -4.70470989
  -11.36734153]
 [ -4.25088032  58.44101386  53.29702765 ... -15.21471059 -12.41288835
  -11.80928333]]


In [47]:
#weight_vector là kq khi nhân chỉnh vị của U và A
weight_vector = np.dot(U.T, A)
print(weight_vector.shape)
print(weight_vector)

(90, 90)
[[-2.47538570e+06 -2.53252487e+07 -2.39880281e+07 ...  1.01758170e+06
   6.00666028e+06 -9.17021892e+06]
 [ 5.40468600e+06  3.76319556e+07  9.82391811e+06 ... -1.37367180e+07
   2.08632397e+06 -1.14893930e+07]
 [-1.80557731e+07 -1.34758995e+07 -1.77884776e+07 ...  3.65533464e+05
   3.20411261e+06 -5.99984216e+05]
 ...
 [-7.36141606e+03 -2.95672916e+05  5.92955615e+04 ... -3.35966061e+05
   5.93245194e+04 -2.83865194e+05]
 [ 2.62452323e+03  2.18881571e+05  1.15846393e+05 ... -2.84112162e+05
   4.34271366e+05 -2.43368136e+04]
 [-4.72576834e+04 -2.15682008e+05 -4.04875206e+04 ...  3.28453136e+04
   3.21321675e+04  5.49856292e+05]]


In [48]:
#tính overall mean bằng cách tính mean của ma trận weight vừa tính
overall_mean = np.mean(weight_vector, axis = 1)
overall_mean = overall_mean.reshape(overall_mean.shape[0],1)
print(overall_mean.shape)

(90, 1)


In [49]:
#tìm within class Sw
#tìm mean cho từng class riêng lẻ
#số class là 15 (vì tập dữ liệu có 15 người)
SW = np.zeros([90,90])
for i in range(15):
    ind = i * 6
    V = weight_vector[:,ind:ind+6]
    mean_local = np.mean(V, axis = 1)
    mean_local = mean_local.reshape(mean_local.shape[0],1)
    mean = np.repeat(mean_local, 6,axis = 1)
    diff = V - mean
    variance = np.dot(diff, diff.T)
    SW = SW + variance

In [50]:
#tìm between class Sb
SB = np.zeros([90,90])
print(variance.shape)
for i in range(15):
    j = i+6
    V = weight_vector[:,i:j]
    mean_local = np.mean(V, axis = 1)
    mean_local = mean_local.reshape(mean_local.shape[0],1)
    diff = mean_local - overall_mean
    sigma = np.dot(diff, mean_local.T)
    SB = SB  + sigma

(90, 90)


In [51]:
#finding the criterion function
#maximises between class (Sb) và minimizes within class (Sw)
J = np.dot(np.linalg.pinv(SW), SB)
print(J.shape)

(90, 90)


In [52]:
#tìm eigenvalues và eigenvectors của ma trận J
eigenval, eigenvec = np.linalg.eig(J)
fisher_faces = np.dot(eigenvec.T, weight_vector)
print(fisher_faces.shape)

(90, 90)


In [53]:
#Bắt đầu công đoạn test
#thực hiện tương tự như tập train_set
x_range = test_set.shape[0]
y_range = test_set.shape[1]
print(M)
print(test_set)
for i in range(x_range):
    for j in range(y_range):
        test_set[i][j] -= M[i]
print(test_set.shape)
print(test_set)

[92.02222222 92.2        92.3        ... 59.11111111 58.32222222
 58.37777778]
[[ 41  44  41 ...  45 120 120]
 [ 45  43  41 ...  41 120 120]
 [ 46  37  43 ...  39 120 122]
 ...
 [ 36  44  42 ...  71  74  73]
 [ 37  41  43 ...  23  75  75]
 [ 37  36  41 ...  37  80  76]]
(10304, 60)
[[-51 -48 -51 ... -47  27  27]
 [-47 -49 -51 ... -51  27  27]
 [-46 -55 -49 ... -53  27  29]
 ...
 [-23 -15 -17 ...  11  14  13]
 [-21 -17 -15 ... -35  16  16]
 [-21 -22 -17 ... -21  21  17]]


In [54]:
weight_vector_test = np.dot(U.T, test_set)
print(weight_vector_test.shape)
print(weight_vector_test)

(90, 60)
[[-1.71624202e+07 -3.00463454e+07 -2.35298344e+07 ...  2.44416330e+06
  -4.67404113e+05  7.19894795e+06]
 [ 1.78243365e+07  1.48528891e+07  2.89573479e+07 ... -1.38414493e+07
  -1.25861515e+06  4.07314154e+06]
 [-1.85701017e+07 -4.02674056e+06 -3.01547485e+06 ... -1.00611024e+06
   2.94821067e+06  4.39267217e+06]
 ...
 [-1.11853143e+04  4.62677497e+04  9.16028410e+04 ... -1.70532107e+04
   6.13272049e+04 -3.12774894e+04]
 [ 1.92726353e+04 -1.15301651e+04  3.73473138e+04 ...  1.33705601e+05
  -8.28071931e+04 -1.51738794e+05]
 [ 5.86925832e+04 -1.15380744e+05  8.61375640e+04 ... -5.26906624e+04
   3.04536083e+04 -4.91769175e+04]]


In [55]:
projected_fisher_faces = np.dot(eigenvec.T, weight_vector_test)
print(projected_fisher_faces.shape)

(90, 60)


In [56]:
#finding the norm between each projected_fisher_faces with that of fisher_faces and then finding the best match
validation = []
count = 0
for i in range(60) : 
    ith_wv = projected_fisher_faces[:,i]
    ans = 0
    index = 0
    for j in range(90):
        jth_wv = fisher_faces[:,j]
        diff = ith_wv - jth_wv
        diff = np.absolute(diff)
        sm = np.sum(diff)
        if ans == 0 :
            ans = sm
            index = j
        else :
            if sm < ans:
                ans = sm
                index = j
    if train_set_number[index] == test_set_number[i]:
        count = count + 1
        validation.append(1)
    else:
        validation.append(0)

In [57]:
print(validation)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [58]:
print("accuracy = ", (count*100)/60, "%")

accuracy =  95.0 %
